# Fusion des données Grocery Sales
## Notebook Python pour merger 7 CSV en 1 table denormalisée

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Chemin des CSV
csv_path = './'  # Chemin local vers les fichiers CSV

# Charger les 7 tables
categories = pd.read_csv(f'{csv_path}categories.csv')
cities = pd.read_csv(f'{csv_path}cities.csv')
countries = pd.read_csv(f'{csv_path}countries.csv')
customers = pd.read_csv(f'{csv_path}customers.csv')
employees = pd.read_csv(f'{csv_path}employees.csv')
products = pd.read_csv(f'{csv_path}products.csv')
sales = pd.read_csv(f'{csv_path}sales.csv')

print("✅ Toutes les tables chargées")
print(f"\nSales shape: {sales.shape}")

✅ Toutes les tables chargées

Sales shape: (6758125, 9)


In [2]:
# 0.1 NETTOYAGE INITIAL DES DATES

print("\n=== NETTOYAGE DES DATES AVANT FUSION ===")
print(f"Avant: {sales['SalesDate'].head(3).tolist()}")

# Convertir SalesDate et supprimer les millisecondes
sales['SalesDate'] = pd.to_datetime(sales['SalesDate'], errors='coerce')
sales['SalesDate'] = sales['SalesDate'].dt.floor('S')  # Arrondir à la seconde

print(f"Après: {sales['SalesDate'].head(3).tolist()}")
print(f"Nulls détectés: {sales['SalesDate'].isnull().sum()}")

# Séparer la date et l'heure
print("\n=== SÉPARATION DATE / TIME ===")
sales['Date'] = sales['SalesDate'].dt.date
sales['Time'] = sales['SalesDate'].dt.time

print(f"Exemples de Date: {sales['Date'].head(3).tolist()}")
print(f"Exemples de Time: {sales['Time'].head(3).tolist()}")



=== NETTOYAGE DES DATES AVANT FUSION ===
Avant: ['2018-02-05 07:38:25.430', '2018-02-02 16:03:31.150', '2018-05-03 19:31:56.880']
Après: [Timestamp('2018-02-05 07:38:25'), Timestamp('2018-02-02 16:03:31'), Timestamp('2018-05-03 19:31:56')]
Nulls détectés: 67526

=== SÉPARATION DATE / TIME ===
Exemples de Date: [datetime.date(2018, 2, 5), datetime.date(2018, 2, 2), datetime.date(2018, 5, 3)]
Exemples de Time: [datetime.time(7, 38, 25), datetime.time(16, 3, 31), datetime.time(19, 31, 56)]


In [3]:
# 1. FUSION PROGRESSIVE

# Sales + Products
fact_sales = sales.merge(products, on='ProductID', how='left')
print(f"After Products merge: {fact_sales.shape}")

# + Categories
fact_sales = fact_sales.merge(categories, on='CategoryID', how='left')
print(f"After Categories merge: {fact_sales.shape}")

# + Customers
fact_sales = fact_sales.merge(customers, on='CustomerID', how='left')
print(f"After Customers merge: {fact_sales.shape}")

# + Cities (via customer CityID)
fact_sales = fact_sales.merge(cities, on='CityID', how='left')
print(f"After Cities merge: {fact_sales.shape}")

# + Countries
fact_sales = fact_sales.merge(countries, on='CountryID', how='left')
print(f"After Countries merge: {fact_sales.shape}")

# + Employees
fact_sales = fact_sales.merge(employees, left_on='SalesPersonID', right_on='EmployeeID', how='left')
print(f"After Employees merge: {fact_sales.shape}")

print("\n✅ Fusion complète")

After Products merge: (6758125, 19)
After Categories merge: (6758125, 20)
After Customers merge: (6758125, 25)
After Cities merge: (6758125, 28)
After Countries merge: (6758125, 30)
After Employees merge: (6758125, 38)

✅ Fusion complète


In [4]:
# 2. NETTOYAGE & TRANSFORMATION

# Convertir SalesDate en datetime
fact_sales['SalesDate'] = pd.to_datetime(fact_sales['SalesDate'], errors='coerce')

# Option A: Supprimer les lignes avec SalesDate null (67,526 lignes)
print(f"Lignes avant suppression des SalesDate null: {len(fact_sales)}")
print(f"Lignes avec SalesDate null: {fact_sales['SalesDate'].isnull().sum()}")

fact_sales = fact_sales.dropna(subset=['SalesDate'])
print(f"Lignes après suppression: {len(fact_sales)}")

# Extraire dimensions temps
fact_sales['Year'] = fact_sales['SalesDate'].dt.year
fact_sales['Month'] = fact_sales['SalesDate'].dt.month
fact_sales['Quarter'] = fact_sales['SalesDate'].dt.quarter
fact_sales['DayOfWeek'] = fact_sales['SalesDate'].dt.day_name()
fact_sales['Date'] = fact_sales['SalesDate'].dt.date

# Renommer colonnes pour clarté
fact_sales.rename(columns={
    'FirstName_x': 'CustomerFirstName',
    'LastName_x': 'CustomerLastName',
    'FirstName_y': 'EmployeeFirstName',
    'LastName_y': 'EmployeeLastName',
    'Address_x': 'CustomerAddress',
    'Address_y': 'EmployeeAddress',
    'CityName': 'City',
    'CountryName': 'Country'
}, inplace=True)

# Supprimer les doublons de colonnes
cols_to_drop = [col for col in fact_sales.columns if col.endswith('_y')]
fact_sales.drop(columns=cols_to_drop, inplace=True)
fact_sales.columns = [col.replace('_x', '') for col in fact_sales.columns]

print(f"\nShape après transformation: {fact_sales.shape}")
print(f"Colonnes finales: {list(fact_sales.columns)}")

Lignes avant suppression des SalesDate null: 6758125
Lignes avec SalesDate null: 67526
Lignes après suppression: 6690599

Shape après transformation: (6690599, 40)
Colonnes finales: ['SalesID', 'SalesPersonID', 'CustomerID', 'ProductID', 'Quantity', 'Discount', 'TotalPrice', 'SalesDate', 'TransactionNumber', 'Date', 'Time', 'ProductName', 'Price', 'CategoryID', 'Class', 'ModifyDate', 'Resistant', 'IsAllergic', 'VitalityDays', 'CategoryName', 'CustomerFirstName', 'MiddleInitial', 'CustomerLastName', 'CityID', 'Address', 'City', 'Zipcode', 'CountryID', 'Country', 'CountryCode', 'EmployeeID', 'EmployeeFirstName', 'EmployeeLastName', 'BirthDate', 'Gender', 'HireDate', 'Year', 'Month', 'Quarter', 'DayOfWeek']


In [5]:
# 2.1. VÉRIFIER ET NETTOYER LES NOMS DE COLONNES

print("\n=== DIAGNOSTIC DES COLONNES ===")
print(f"\nTotal colonnes: {len(fact_sales.columns)}")

# Vérifier les problèmes courants
issues = []

for col in fact_sales.columns:
    problems = []
    
    # Vérifier espaces au début/fin
    if col != col.strip():
        problems.append("espaces début/fin")
    
    # Vérifier espaces au milieu
    if ' ' in col:
        problems.append("contient espaces")
    
    # Vérifier caractères spéciaux
    if any(c in col for c in ['/', '\\', '-', '.', '!', '@', '#', '$', '%']):
        problems.append("caractères spéciaux")
    
    # Vérifier tirets bas doubles
    if '__' in col:
        problems.append("tirets bas doubles")
    
    if problems:
        issues.append(f"  ❌ {col} → {', '.join(problems)}")

# Vérifier doublons
duplicates = fact_sales.columns[fact_sales.columns.duplicated()].tolist()
if duplicates:
    issues.append(f"  ❌ Colonnes dupliquées: {duplicates}")

# Afficher résultats
if issues:
    print("\n🔴 PROBLÈMES DÉTECTÉS:")
    for issue in issues:
        print(issue)
else:
    print("\n✅ AUCUN PROBLÈME DÉTECTÉ")

# Afficher la liste complète des colonnes
print(f"\n📋 LISTE DES COLONNES ({len(fact_sales.columns)}):")
for i, col in enumerate(fact_sales.columns, 1):
    dtype = fact_sales[col].dtype
    null_count = fact_sales[col].isnull().sum()
    print(f"  {i:2d}. {col:30s} ({str(dtype):15s}) | Nulls: {null_count:,}")



=== DIAGNOSTIC DES COLONNES ===

Total colonnes: 40

✅ AUCUN PROBLÈME DÉTECTÉ

📋 LISTE DES COLONNES (40):
   1. SalesID                        (int64          ) | Nulls: 0
   2. SalesPersonID                  (int64          ) | Nulls: 0
   3. CustomerID                     (int64          ) | Nulls: 0
   4. ProductID                      (int64          ) | Nulls: 0
   5. Quantity                       (int64          ) | Nulls: 0
   6. Discount                       (float64        ) | Nulls: 0
   7. TotalPrice                     (float64        ) | Nulls: 0
   8. SalesDate                      (datetime64[ns] ) | Nulls: 0
   9. TransactionNumber              (object         ) | Nulls: 0
  10. Date                           (object         ) | Nulls: 0
  11. Time                           (object         ) | Nulls: 0
  12. ProductName                    (object         ) | Nulls: 0
  13. Price                          (float64        ) | Nulls: 0
  14. CategoryID                   

In [6]:
# 2.2. ANALYSE DES FORMATS DE DONNÉES

print("\n=== ANALYSE DES FORMATS DE DONNÉES ===")

# Résumé par type de données
print("\n📊 RÉSUMÉ PAR TYPE DE DONNÉES:")
dtype_counts = fact_sales.dtypes.value_counts()
for dtype, count in dtype_counts.items():
    print(f"  {str(dtype):20s}: {count:2d} colonnes")

# Détails pour chaque colonne
print(f"\n📋 DÉTAILS DES COLONNES ({len(fact_sales.columns)}):")
print(f"{'N°':>3} {'Nom Colonne':30s} {'Type':20s} {'Nulls':>8} {'Min/Max ou Exemples':40s}")
print("─" * 105)

for i, col in enumerate(fact_sales.columns, 1):
    dtype = fact_sales[col].dtype
    null_count = fact_sales[col].isnull().sum()
    
    # Obtenir des informations sur la plage ou les exemples
    if pd.api.types.is_numeric_dtype(fact_sales[col]):
        min_val = fact_sales[col].min()
        max_val = fact_sales[col].max()
        info = f"[{min_val} à {max_val}]"
    elif pd.api.types.is_datetime64_any_dtype(fact_sales[col]):
        min_val = fact_sales[col].min()
        max_val = fact_sales[col].max()
        info = f"[{min_val} à {max_val}]"
    elif pd.api.types.is_object_dtype(fact_sales[col]):
        # Exemple de valeur
        sample = fact_sales[col].dropna().iloc[0] if fact_sales[col].notna().any() else "N/A"
        info = f"Ex: '{str(sample)[:30]}'"
    else:
        info = str(dtype)
    
    print(f"{i:3d} {col:30s} {str(dtype):20s} {null_count:8,d} {info:40s}")

# Statistiques pour colonnes numériques
print("\n\n📈 STATISTIQUES DES COLONNES NUMÉRIQUES:")
numeric_cols = fact_sales.select_dtypes(include=[np.number]).columns
print(fact_sales[numeric_cols].describe().to_string())



=== ANALYSE DES FORMATS DE DONNÉES ===

📊 RÉSUMÉ PAR TYPE DE DONNÉES:
  object              : 22 colonnes
  int64               : 10 colonnes
  float64             :  4 colonnes
  int32               :  3 colonnes
  datetime64[ns]      :  1 colonnes

📋 DÉTAILS DES COLONNES (40):
 N° Nom Colonne                    Type                    Nulls Min/Max ou Exemples                     
─────────────────────────────────────────────────────────────────────────────────────────────────────────
  1 SalesID                        int64                       0 [1 à 6758125]                           
  2 SalesPersonID                  int64                       0 [1 à 23]                                
  3 CustomerID                     int64                       0 [1 à 98759]                             
  4 ProductID                      int64                       0 [1 à 452]                               
  5 Quantity                       int64                       0 [1 à 25]          

In [7]:
# 3. APERÇU DE LA TABLE FUSIONNÉE

print("\n=== APERÇU DE LA TABLE DENORMALISÉE ===")
print(fact_sales.head(3).to_string())
print(f"\nTotal lignes: {len(fact_sales)}")
print(f"Total colonnes: {len(fact_sales.columns)}")
print(f"\nValeurs nulles:\n{fact_sales.isnull().sum()}")


=== APERÇU DE LA TABLE DENORMALISÉE ===
   SalesID  SalesPersonID  CustomerID  ProductID  Quantity  Discount  TotalPrice           SalesDate     TransactionNumber        Date      Time              ProductName    Price  CategoryID   Class               ModifyDate Resistant IsAllergic  VitalityDays CategoryName CustomerFirstName MiddleInitial CustomerLastName  CityID                   Address         City  Zipcode  CountryID        Country CountryCode  EmployeeID EmployeeFirstName EmployeeLastName                BirthDate Gender                 HireDate  Year  Month  Quarter DayOfWeek
0        1              6       27039        381         7       0.0         0.0 2018-02-05 07:38:25  FQL4S94E4ME1EZFTG42G  2018-02-05  07:38:25         Vaccum Bag 10x13  44.2337           1    High  2018-01-06 22:26:53.580   Unknown    Unknown          41.0  Confections             Susan             V            Green      54  826 Rocky Second Freeway  Albuquerque    55358         32  United States      

In [8]:
# 3.1. VÉRIFIER LE FORMAT DE SALESDATE

print("\n=== ANALYSE DE SALESDATE ===")
print(f"Type de colonne: {fact_sales['SalesDate'].dtype}")
print(f"\nExemples de valeurs SalesDate:")
print(fact_sales['SalesDate'].head(10))
print(f"\nValeurs nulles dans SalesDate: {fact_sales['SalesDate'].isnull().sum()}")
print(f"Valeurs NaN: {fact_sales['SalesDate'].isna().sum()}")
print(f"\nValeurs uniques (premiers 5):")
print(fact_sales['SalesDate'].unique()[:5])



=== ANALYSE DE SALESDATE ===
Type de colonne: datetime64[ns]

Exemples de valeurs SalesDate:
0   2018-02-05 07:38:25
1   2018-02-02 16:03:31
2   2018-05-03 19:31:56
3   2018-04-07 14:43:55
4   2018-02-12 15:37:03
5   2018-02-07 10:33:24
6   2018-03-02 23:09:58
7   2018-01-17 13:41:38
8   2018-04-27 06:19:58
9   2018-03-26 22:12:08
Name: SalesDate, dtype: datetime64[ns]

Valeurs nulles dans SalesDate: 0
Valeurs NaN: 0

Valeurs uniques (premiers 5):
<DatetimeArray>
['2018-02-05 07:38:25', '2018-02-02 16:03:31', '2018-05-03 19:31:56',
 '2018-04-07 14:43:55', '2018-02-12 15:37:03']
Length: 5, dtype: datetime64[ns]


In [9]:
# 4. ANALYSES POST-FUSION

print("\n" + "="*50)
print("ANALYSES - AFTER FUSION")
print("="*50)

# A. CA Total par mois
print("\n📊 CA TOTAL PAR MOIS:")
ca_par_mois = fact_sales.groupby(['Year', 'Month'])['TotalPrice'].sum().reset_index()
ca_par_mois['YearMonth'] = ca_par_mois['Year'].astype(str) + '-' + ca_par_mois['Month'].astype(str).str.zfill(2)
print(ca_par_mois[['YearMonth', 'TotalPrice']].to_string(index=False))

# B. Top 10 produits par CA
print("\n🏆 TOP 10 PRODUITS PAR CA:")
top_products = fact_sales.groupby('ProductName').agg({
    'TotalPrice': 'sum',
    'Quantity': 'sum'
}).sort_values('TotalPrice', ascending=False).head(10)
print(top_products.to_string())

# C. Top 10 clients par CA
print("\n👥 TOP 10 CLIENTS PAR CA:")
top_clients = fact_sales.groupby(['CustomerID', 'CustomerFirstName', 'CustomerLastName']).agg({
    'TotalPrice': 'sum',
    'Quantity': 'sum',
    'SalesID': 'count'
}).sort_values('TotalPrice', ascending=False).head(10)
top_clients.rename(columns={'SalesID': 'NbTransactions'}, inplace=True)
print(top_clients.to_string())

# D. Top 5 vendeurs
print("\n💼 TOP 5 VENDEURS PAR CA:")
top_sellers = fact_sales.groupby(['EmployeeFirstName', 'EmployeeLastName']).agg({
    'TotalPrice': 'sum',
    'Quantity': 'sum',
    'SalesID': 'count'
}).sort_values('TotalPrice', ascending=False).head(5)
top_sellers.rename(columns={'SalesID': 'NbVentes'}, inplace=True)
print(top_sellers.to_string())


ANALYSES - AFTER FUSION

📊 CA TOTAL PAR MOIS:
YearMonth  TotalPrice
  2018-01         0.0
  2018-02         0.0
  2018-03         0.0
  2018-04         0.0
  2018-05         0.0

🏆 TOP 10 PRODUITS PAR CA:
                            TotalPrice  Quantity
ProductName                                     
Anchovy Paste - 56 G Tube          0.0    192849
Pickerel - Fillets                 0.0    193837
Rum - Coconut, Malibu              0.0    192375
Rosemary - Primerba, Paste         0.0    193059
Rosemary - Dry                     0.0    193395
Rice - Long Grain                  0.0    193599
Rice - Jasmine Sented              0.0    191538
Remy Red                           0.0    190772
Raspberries - Fresh                0.0    190251
Rambutan                           0.0    194779

👥 TOP 10 CLIENTS PAR CA:
                                               TotalPrice  Quantity  NbTransactions
CustomerID CustomerFirstName CustomerLastName                                      
1          S

In [10]:
# E. CA par Géographie
print("\n🌍 CA PAR PAYS:")
ca_pays = fact_sales.groupby('Country')['TotalPrice'].sum().sort_values(ascending=False)
print(ca_pays.to_string())

print("\n🏙️  CA PAR VILLE (TOP 10):")
ca_ville = fact_sales.groupby('City')['TotalPrice'].sum().sort_values(ascending=False).head(10)
print(ca_ville.to_string())

# F. CA par Catégorie
print("\n📦 CA PAR CATÉGORIE:")
ca_categorie = fact_sales.groupby('CategoryName').agg({
    'TotalPrice': 'sum',
    'Quantity': 'sum'
}).sort_values('TotalPrice', ascending=False)
print(ca_categorie.to_string())


🌍 CA PAR PAYS:
Country
United States    0.0

🏙️  CA PAR VILLE (TOP 10):
City
Akron           0.0
Albuquerque     0.0
Raleigh         0.0
Portland        0.0
Pittsburgh      0.0
Phoenix         0.0
Philadelphia    0.0
Omaha           0.0
Oklahoma        0.0
Oakland         0.0

📦 CA PAR CATÉGORIE:
              TotalPrice  Quantity
CategoryName                      
Beverages            0.0   7320055
Cereals              0.0   8647338
Confections          0.0  10967277
Dairy                0.0   6745711
Grain                0.0   5378584
Meat                 0.0   9621629
Poultry              0.0   9071479
Produce              0.0   8283590
Seafood              0.0   6925812
Shell fish           0.0   6914002
Snails               0.0   7128070


In [11]:
# G. Statistiques globales
print("\n" + "="*50)
print("📈 STATISTIQUES GLOBALES")
print("="*50)

print(f"\nCA Total: ${fact_sales['TotalPrice'].sum():,.2f}")
print(f"Quantité totale vendue: {fact_sales['Quantity'].sum():,} unités")
print(f"Ticket moyen: ${fact_sales['TotalPrice'].mean():,.2f}")
print(f"Discount total appliqué: ${fact_sales['Discount'].sum():,.2f}")
print(f"\nNombre de ventes: {len(fact_sales):,}")
print(f"Nombre de clients uniques: {fact_sales['CustomerID'].nunique()}")
print(f"Nombre de vendeurs: {fact_sales['EmployeeID'].nunique()}")
print(f"Nombre de produits: {fact_sales['ProductID'].nunique()}")
print(f"Nombre de villes: {fact_sales['CityID'].nunique()}")
print(f"Période: {fact_sales['Date'].min()} à {fact_sales['Date'].max()}")


📈 STATISTIQUES GLOBALES

CA Total: $0.00
Quantité totale vendue: 87,003,547 unités
Ticket moyen: $0.00
Discount total appliqué: $200,481.00

Nombre de ventes: 6,690,599
Nombre de clients uniques: 98759
Nombre de vendeurs: 23
Nombre de produits: 452
Nombre de villes: 96
Période: 2018-01-01 à 2018-05-09


In [12]:
# H. Vérification des doublons et anomalies
print("\n" + "="*50)
print("🔍 CONTRÔLE QUALITÉ")
print("="*50)

print(f"\nDoublons dans les ventes: {fact_sales.duplicated(subset=['SalesID']).sum()}")
print(f"Ventes avec TotalPrice = 0: {(fact_sales['TotalPrice'] == 0).sum()}")
print(f"Ventes avec Quantity <= 0: {(fact_sales['Quantity'] <= 0).sum()}")
print(f"Valeurs nulles critiques:")
critical_cols = ['SalesID', 'ProductID', 'CustomerID', 'EmployeeID', 'TotalPrice']
for col in critical_cols:
    null_count = fact_sales[col].isnull().sum()
    print(f"  - {col}: {null_count}")


🔍 CONTRÔLE QUALITÉ

Doublons dans les ventes: 0
Ventes avec TotalPrice = 0: 6690599
Ventes avec Quantity <= 0: 0
Valeurs nulles critiques:
  - SalesID: 0
  - ProductID: 0
  - CustomerID: 0
  - EmployeeID: 0
  - TotalPrice: 0


In [13]:
# 5. PRÊT POUR TALEND
print("\n" + "="*50)
print("✅ TABLE DENORMALISÉE PRÊTE POUR TALEND")
print("="*50)

print(f"\nInfo table finale:")
print(f"  - Lignes: {fact_sales.shape[0]:,}")
print(f"  - Colonnes: {fact_sales.shape[1]}")
print(f"\nColonnes disponibles:")
for i, col in enumerate(fact_sales.columns, 1):
    dtype = fact_sales[col].dtype
    print(f"  {i:2d}. {col:30s} ({dtype})")

print("\n✨ Prêt pour Talend! (df = fact_sales)")


✅ TABLE DENORMALISÉE PRÊTE POUR TALEND

Info table finale:
  - Lignes: 6,690,599
  - Colonnes: 40

Colonnes disponibles:
   1. SalesID                        (int64)
   2. SalesPersonID                  (int64)
   3. CustomerID                     (int64)
   4. ProductID                      (int64)
   5. Quantity                       (int64)
   6. Discount                       (float64)
   7. TotalPrice                     (float64)
   8. SalesDate                      (datetime64[ns])
   9. TransactionNumber              (object)
  10. Date                           (object)
  11. Time                           (object)
  12. ProductName                    (object)
  13. Price                          (float64)
  14. CategoryID                     (int64)
  15. Class                          (object)
  16. ModifyDate                     (object)
  17. Resistant                      (object)
  18. IsAllergic                     (object)
  19. VitalityDays                   (float64

In [14]:
# 5.5. NETTOYAGE AVANT EXPORT - CRÉER DateID ET SUPPRIMER COLONNES TEMPORAIRES

print("\n" + "="*50)
print("🧹 NETTOYAGE AVANT EXPORT")
print("="*50)

# Créer DateID (format YYYYMMDD) à partir de Date
fact_sales['DateID'] = pd.to_datetime(fact_sales['Date']).dt.strftime('%Y%m%d').astype(int)

print(f"\n✅ Colonne DateID créée")
print(f"   Exemples: {fact_sales['DateID'].head(3).tolist()}")

# Supprimer les colonnes temporaires
cols_to_drop = ['Year', 'Month', 'Quarter', 'DayOfWeek']
fact_sales.drop(columns=cols_to_drop, inplace=True)

print(f"\n✅ Colonnes supprimées: {cols_to_drop}")
print(f"\n📊 Table finale:")
print(f"   Lignes: {fact_sales.shape[0]:,}")
print(f"   Colonnes: {fact_sales.shape[1]}")
print(f"\n📋 Colonnes restantes:")
for i, col in enumerate(fact_sales.columns, 1):
    print(f"   {i:2d}. {col}")



🧹 NETTOYAGE AVANT EXPORT

✅ Colonne DateID créée
   Exemples: [20180205, 20180202, 20180503]

✅ Colonnes supprimées: ['Year', 'Month', 'Quarter', 'DayOfWeek']

📊 Table finale:
   Lignes: 6,690,599
   Colonnes: 37

📋 Colonnes restantes:
    1. SalesID
    2. SalesPersonID
    3. CustomerID
    4. ProductID
    5. Quantity
    6. Discount
    7. TotalPrice
    8. SalesDate
    9. TransactionNumber
   10. Date
   11. Time
   12. ProductName
   13. Price
   14. CategoryID
   15. Class
   16. ModifyDate
   17. Resistant
   18. IsAllergic
   19. VitalityDays
   20. CategoryName
   21. CustomerFirstName
   22. MiddleInitial
   23. CustomerLastName
   24. CityID
   25. Address
   26. City
   27. Zipcode
   28. CountryID
   29. Country
   30. CountryCode
   31. EmployeeID
   32. EmployeeFirstName
   33. EmployeeLastName
   34. BirthDate
   35. Gender
   36. HireDate
   37. DateID


In [15]:
fact_sales.columns

Index(['SalesID', 'SalesPersonID', 'CustomerID', 'ProductID', 'Quantity',
       'Discount', 'TotalPrice', 'SalesDate', 'TransactionNumber', 'Date',
       'Time', 'ProductName', 'Price', 'CategoryID', 'Class', 'ModifyDate',
       'Resistant', 'IsAllergic', 'VitalityDays', 'CategoryName',
       'CustomerFirstName', 'MiddleInitial', 'CustomerLastName', 'CityID',
       'Address', 'City', 'Zipcode', 'CountryID', 'Country', 'CountryCode',
       'EmployeeID', 'EmployeeFirstName', 'EmployeeLastName', 'BirthDate',
       'Gender', 'HireDate', 'DateID'],
      dtype='object')

In [16]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

fact_sales.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6690599 entries, 0 to 6758124
Data columns (total 37 columns):
 #   Column             Dtype         
---  ------             -----         
 0   SalesID            int64         
 1   SalesPersonID      int64         
 2   CustomerID         int64         
 3   ProductID          int64         
 4   Quantity           int64         
 5   Discount           float64       
 6   TotalPrice         float64       
 7   SalesDate          datetime64[ns]
 8   TransactionNumber  object        
 9   Date               object        
 10  Time               object        
 11  ProductName        object        
 12  Price              float64       
 13  CategoryID         int64         
 14  Class              object        
 15  ModifyDate         object        
 16  Resistant          object        
 17  IsAllergic         object        
 18  VitalityDays       float64       
 19  CategoryName       object        
 20  CustomerFirstName  object    

In [ ]:
# 6. EXPORTER EN CSV

print("\n" + "="*50)
print("💾 EXPORT EN CSV")
print("="*50)

# Définir le chemin d'export
output_file = './grocery_sales_denormalized.csv'

# Exporter en CSV
fact_sales.to_csv(output_file, index=False, encoding='utf-8')

print(f"\n✅ Table exportée avec succès!")
print(f"   Fichier: {output_file}")
print(f"   Lignes: {len(fact_sales):,}")
print(f"   Colonnes: {len(fact_sales.columns)}")
print(f"   Taille: {fact_sales.memory_usage(deep=True).sum() / 1024**2:.2f} MB")



💾 EXPORT EN CSV

✅ Table exportée avec succès!
   Fichier: ./grocery_sales_denormalized.csv
   Lignes: 6,690,599
   Colonnes: 37


In [ ]:
fact_sales = pd.read_csv(f'{csv_path}grocery_sales_denormalized.csv')


In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

fact_sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6690599 entries, 0 to 6690598
Data columns (total 40 columns):
 #   Column             Dtype  
---  ------             -----  
 0   SalesID            int64  
 1   SalesPersonID      int64  
 2   CustomerID         int64  
 3   ProductID          int64  
 4   Quantity           int64  
 5   Discount           float64
 6   TotalPrice         float64
 7   SalesDate          object 
 8   TransactionNumber  object 
 9   Date               object 
 10  Time               object 
 11  ProductName        object 
 12  Price              float64
 13  CategoryID         int64  
 14  Class              object 
 15  ModifyDate         object 
 16  Resistant          object 
 17  IsAllergic         object 
 18  VitalityDays       float64
 19  CategoryName       object 
 20  CustomerFirstName  object 
 21  MiddleInitial      object 
 22  CustomerLastName   object 
 23  CityID             int64  
 24  Address            object 
 25  City              

In [ ]:
# 6.2. VÉRIFIER LE SÉPARATEUR DU CSV

print("\n" + "="*50)
print("🔍 VÉRIFICATION DU SÉPARATEUR CSV")
print("="*50)

# Lire les premières lignes du fichier CSV
with open(output_file, 'r', encoding='utf-8') as f:
    print("\n📄 PREMIÈRES LIGNES DU FICHIER:")
    for i in range(3):
        line = f.readline().strip()
        print(f"Ligne {i+1}: {line[:100]}...")

# Afficher info du séparateur
print("\n📊 SÉPARATEUR UTILISÉ:")
print(f"  Séparateur actuel: , (virgule)")
print(f"  Si vous préférez ;  (point-virgule), exécutez la cellule suivante")

# Afficher le nombre de champs dans la première ligne
with open(output_file, 'r', encoding='utf-8') as f:
    first_line = f.readline()
    nb_fields_comma = len(first_line.split(','))
    nb_fields_semicolon = len(first_line.split(';'))
    print(f"\n  Avec virgule: {nb_fields_comma} champs")
    print(f"  Avec point-virgule: {nb_fields_semicolon} champs")
    print(f"  ➜ Le bon séparateur est la virgule ✅")



🔍 VÉRIFICATION DU SÉPARATEUR CSV

📄 PREMIÈRES LIGNES DU FICHIER:
Ligne 1: SalesID,SalesPersonID,CustomerID,ProductID,Quantity,Discount,TotalPrice,SalesDate,TransactionNumber,...
Ligne 2: 1,6,27039,381,7,0.0,0.0,2018-02-05 07:38:25,FQL4S94E4ME1EZFTG42G,2018-02-05,07:38:25,Vaccum Bag 10x1...
Ligne 3: 2,16,25011,61,7,0.0,0.0,2018-02-02 16:03:31,12UGLX40DJ1A5DTFBHB8,2018-02-02,16:03:31,Sardines,62.546...

📊 SÉPARATEUR UTILISÉ:
  Séparateur actuel: , (virgule)
  Si vous préférez ;  (point-virgule), exécutez la cellule suivante

  Avec virgule: 40 champs
  Avec point-virgule: 1 champs
  ➜ Le bon séparateur est la virgule ✅
